In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, col, window, avg, round as _round

spark = (
    SparkSession.builder
    .appName("Lab2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [5]:
df = spark.read.json("transactions_10k.jsonl")
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

In [13]:
#zad 2.2
from pyspark.sql.functions import min as _min, max as _max, sum as _sum, round as _round

(df.groupBy("category").agg(
        _round(_sum("amount"), 2).alias("suma PLN"),
        _round(_min("amount"), 2).alias("min PLN"),
        _round(_max("amount"), 2).alias("max PLN"))
    .orderBy("category")
    .show()
)

+-----------+----------+-------+-------+
|   category|  suma PLN|min PLN|max PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [21]:
#zad 3.2
from pyspark.sql.functions import count, sum as _sum, round as _round

(
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba tx"),
        _round(_sum("amount"), 2).alias("suma PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba tx",
        "suma PLN"
    )
    .orderBy("od", "store")
    .show(truncate=False)
)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba tx|suma PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [23]:
#zad3.3
from pyspark.sql.functions import desc

(
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(_sum("amount"), 2).alias("suma PLN"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "suma PLN"
    )
    .orderBy(desc("suma PLN"))
    .limit(1)
    .show(truncate=False)
)

+-------------------+-------------------+---------+
|od                 |do                 |suma PLN |
+-------------------+-------------------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|483309.86|
+-------------------+-------------------+---------+



In [ ]:
#zad 4.2
#Sliding ma więcej rekordów, ponieważ okna nachodzą na siebie,
#więc jedna transakcja może zostać przypisana do kilku okien jednocześnie.
#Okno przesuwa się co 30 minut, mimo że obejmuje pełną godzinę,
#dlatego sąsiednie przedziały częściowo zawierają te same dane.

In [ ]:
#zadanie 5
#1. 4661

#2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#groupBy("store") grupuje dane wyłącznie po sklepie.

#groupBy(window(...), "store") grupuje dane jednocześnie po sklepie
#i oknach czasowych, dlatego pokazuje statystyki w różnych przedziałach czasu.

#3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    2 

In [25]:
#prac. dom. 1

(
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia PLN"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia PLN"
    )
    .orderBy("srednia PLN")
    .limit(1)
    .show(truncate=False)
)

+-------------------+-------------------+-----------+
|od                 |do                 |srednia PLN|
+-------------------+-------------------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01     |
+-------------------+-------------------+-----------+



In [33]:
#

(
    df.filter(
        (col("timestamp") >= "2026-04-12 09:00:00") &
        (col("timestamp") < "2026-04-12 09:30:00")
    )
    .groupBy("category")
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy("category")
    .show(truncate=False)
)

+-----------+---------+
|category   |liczba_tx|
+-----------+---------+
|elektronika|611      |
|książki    |622      |
|odzież     |605      |
|żywność    |567      |
+-----------+---------+



In [34]:
#PD3

(
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx"
    )
    .orderBy(col("liczba_tx").desc())
    .limit(1)
    .show(truncate=False)
)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
+-------------------+-------------------+---------+



In [23]:
spark.stop()